# Lecture 03 · 排序、树与检索

**数据结构与算法 B · 小班讨论课**  
本讲练习聚焦二叉树：如何表示一棵树，如何从遍历序列中恢复结构，如何判断结构性质，以及如何在树上做简单动态规划。

---

## 本节题单

| 题号 | 题目 | 核心知识点 |
|---|---|---|
| [P1030](https://www.luogu.com.cn/problem/P1030) | 求先序排列 | 中序 + 后序还原二叉树 |
| [P1229](https://www.luogu.com.cn/problem/P1229) | 遍历问题 | 前序 + 后序的不唯一性 |
| [P1305](https://www.luogu.com.cn/problem/P1305) | 新二叉树 | 指针式建树与递归遍历 |
| [P5018](https://www.luogu.com.cn/problem/P5018) | 对称二叉树 | 镜像结构、子树哈希 |
| [P1364](https://www.luogu.com.cn/problem/P1364) | 医院设置 | 树上距离和、换根 DP |

---

# 题目一：P1030 求先序排列

[Luogu P1030](https://www.luogu.com.cn/problem/P1030)

## 1.1 题目描述

给出一棵二叉树的**中序遍历**与**后序遍历**，求它的**先序遍历**。

- 节点用互不相同的大写字母表示。
- 节点数不超过 $8$。
- 输入两行：第一行为中序遍历，第二行为后序遍历。
- 输出一行：先序遍历。

## 1.2 思路分析

三种遍历的根节点位置：

| 遍历 | 顺序 |
|---|---|
| 先序 | 根 → 左 → 右 |
| 中序 | 左 → 根 → 右 |
| 后序 | 左 → 右 → 根 |

因此：

1. 后序遍历最后一个字符一定是当前子树的根。
2. 在中序遍历中找到根的位置，就能切开左子树与右子树。
3. 左右子树的节点数量确定后，也能同步切开后序遍历。
4. 对左右子树递归处理，答案为：`根 + 左子树先序 + 右子树先序`。

递归不变量：每次函数接收的两段字符串表示同一棵子树，只是遍历方式不同。

时间复杂度：若每次用 `find` 查找根，最坏 $O(n^2)$；本题 $n \le 8$，完全足够。若预处理位置表，可降到 $O(n)$。

## 1.3 参考代码

```python
def preorder_from_in_post(inorder: str, postorder: str) -> str:
    if not inorder:
        return ""

    root = postorder[-1]
    k = inorder.index(root)

    left_in = inorder[:k]
    right_in = inorder[k + 1:]

    left_post = postorder[:k]
    right_post = postorder[k:-1]

    return root + preorder_from_in_post(left_in, left_post) + preorder_from_in_post(right_in, right_post)


inorder = input().strip()
postorder = input().strip()
print(preorder_from_in_post(inorder, postorder))
```

## 1.4 变体讨论

### 变体 1：已知先序和中序，求后序

核心思想完全对称：先序第一个字符是根；中序切分左右子树；后序答案为 `左 + 右 + 根`。

```python
def postorder_from_pre_in(preorder: str, inorder: str) -> str:
    if not preorder:
        return ""

    root = preorder[0]
    k = inorder.index(root)

    left_pre = preorder[1:1 + k]
    right_pre = preorder[1 + k:]

    left_in = inorder[:k]
    right_in = inorder[k + 1:]

    return postorder_from_pre_in(left_pre, left_in) + postorder_from_pre_in(right_pre, right_in) + root
```

### 变体 2：用下标递归，避免反复切片

当 $n$ 较大时，字符串切片会产生额外开销。更工程化的写法是传递区间下标。

```python
def preorder_from_in_post_fast(inorder: str, postorder: str) -> str:
    pos = {ch: i for i, ch in enumerate(inorder)}
    ans = []

    def solve(il: int, ir: int, pl: int, pr: int) -> None:
        if il > ir:
            return
        root = postorder[pr]
        ans.append(root)

        k = pos[root]
        left_size = k - il

        solve(il, k - 1, pl, pl + left_size - 1)
        solve(k + 1, ir, pl + left_size, pr - 1)

    solve(0, len(inorder) - 1, 0, len(postorder) - 1)
    return "".join(ans)
```

## 1.5 小结

这题最值得记住的是：**中序遍历负责划分左右子树，先序/后序负责定位根节点**。只要根节点和左右规模确定，递归就自然成立。

---

# 题目二：P1229 遍历问题

[Luogu P1229](https://www.luogu.com.cn/problem/P1229)

## 2.1 题目描述

给定一棵二叉树的**前序遍历**和**后序遍历**，求可能的**中序遍历序列数量**。

- 第一行：前序遍历 $s_1$。
- 第二行：后序遍历 $s_2$。
- 字符互不相同，保证至少存在一棵二叉树满足条件。
- 输出可能的中序遍历数量，结果不超过 $2^{63}-1$。

## 2.2 思路分析

前序 + 后序通常**不能唯一确定二叉树**。

为什么？

如果某个节点只有一个孩子，那么这个孩子既可以被看作左孩子，也可以被看作右孩子：

```text
    A          A
   /            \
  B              B
```

这两棵树的前序都是 `AB`，后序都是 `BA`，但中序分别是 `BA` 与 `AB`。

因此，每出现一个“只有一个孩子”的节点，答案就乘以 $2$。

如何识别？

若在前序中出现相邻关系 `A B`，而在后序中出现相邻关系 `B A`，说明存在一种子树形态：`A` 的唯一孩子子树根为 `B`。这样的节点带来一次左右选择。

设这样的次数为 $k$，答案为：

$$
2^k
$$

## 2.3 参考代码

```python
pre = input().strip()
post = input().strip()
n = len(pre)

ambiguous = 0

for i in range(n - 1):
    for j in range(n - 1):
        if pre[i] == post[j + 1] and pre[i + 1] == post[j]:
            ambiguous += 1

print(1 << ambiguous)
```

复杂度：$O(n^2)$。本题字符串长度较小，这种写法最直观。若需要优化，可建立后序相邻对集合，将复杂度降为 $O(n)$。

## 2.4 变体讨论

### 变体 1：判断前序 + 后序是否能唯一确定二叉树

唯一当且仅当不存在只有一个孩子的节点，即上面的 `ambiguous == 0`。

```python
def is_unique_from_pre_post(pre: str, post: str) -> bool:
    pairs = {(post[i + 1], post[i]) for i in range(len(post) - 1)}
    for i in range(len(pre) - 1):
        if (pre[i], pre[i + 1]) in pairs:
            return False
    return True
```

### 变体 2：构造一种可能的中序遍历

当前序 + 后序不唯一时，可以约定：遇到唯一孩子，一律当作左孩子。这样能构造出一种合法中序遍历。

```python
def one_inorder_from_pre_post(pre: str, post: str) -> str:
    if not pre:
        return ""
    if len(pre) == 1:
        return pre

    root = pre[0]
    left_root = pre[1]
    k = post.index(left_root)
    left_size = k + 1

    left_pre = pre[1:1 + left_size]
    left_post = post[:left_size]
    right_pre = pre[1 + left_size:]
    right_post = post[left_size:-1]

    return one_inorder_from_pre_post(left_pre, left_post) + root + one_inorder_from_pre_post(right_pre, right_post)
```

## 2.5 小结

P1030 讲的是“如何恢复”；P1229 讲的是“什么时候恢复不了”。前序和后序都能看到根，但都无法告诉我们唯一孩子在左边还是右边，这正是不唯一性的来源。

---

# 题目三：P1305 新二叉树

[Luogu P1305](https://www.luogu.com.cn/problem/P1305)

## 3.1 题目描述

输入一棵二叉树，输出其前序遍历。

- 第一行：节点数 $n$，$1 \le n \le 26$。
- 接下来 $n$ 行，每行三个字符：`root left right`。
- `left` 与 `right` 为该节点的左右孩子，空节点用 `*` 表示。
- 保证第一行读入的节点是整棵树的根。

## 3.2 思路分析

这题不需要从遍历序列恢复结构，而是直接给出每个节点的左右孩子。

我们可以用字典保存：

```text
children[x] = (left_child, right_child)
```

然后从根节点开始递归前序遍历：

1. 访问当前节点。
2. 递归访问左子树。
3. 递归访问右子树。

边界：遇到 `*` 直接返回。

## 3.3 参考代码

```python
n = int(input())
children = {}
root = None

for i in range(n):
    line = input().strip()
    x, left, right = line[0], line[1], line[2]
    if i == 0:
        root = x
    children[x] = (left, right)

ans = []

def preorder(x: str) -> None:
    if x == '*':
        return
    ans.append(x)
    left, right = children[x]
    preorder(left)
    preorder(right)

preorder(root)
print("".join(ans))
```

复杂度：每个节点访问一次，时间 $O(n)$，空间 $O(n)$。

## 3.4 变体讨论

### 变体 1：同时输出前序、中序、后序

同一棵树，只改变“访问根节点”的时机。

```python
def traverse_all(root: str, children: dict[str, tuple[str, str]]) -> tuple[str, str, str]:
    pre, ino, post = [], [], []

    def dfs(x: str) -> None:
        if x == '*':
            return
        left, right = children[x]
        pre.append(x)
        dfs(left)
        ino.append(x)
        dfs(right)
        post.append(x)

    dfs(root)
    return "".join(pre), "".join(ino), "".join(post)
```

### 变体 2：由带空标记的先序串还原二叉树

若输入形如 `ABD**E**C**`，其中 `*` 表示空节点，则可以按先序递归读取。

```python
def build_from_preorder_with_null(s: str):
    idx = 0

    def build():
        nonlocal idx
        ch = s[idx]
        idx += 1
        if ch == '*':
            return None
        left = build()
        right = build()
        return (ch, left, right)

    return build()


def preorder_tuple(node) -> str:
    if node is None:
        return ""
    value, left, right = node
    return value + preorder_tuple(left) + preorder_tuple(right)
```

## 3.5 小结

这题是二叉树表示法的基础题。关键不是算法复杂，而是把“输入描述”稳定转换为“左右孩子关系”，之后遍历就只是模板。

---

# 题目四：P5018 对称二叉树

[Luogu P5018](https://www.luogu.com.cn/problem/P5018)

## 4.1 题目描述

给定一棵带点权的有根二叉树，找出节点数最多的对称二叉子树，并输出它的节点数。

一棵有根二叉树是对称的，当且仅当：

1. 左右结构互为镜像；
2. 镜像位置上的点权相等。

输入：

- 第一行：节点数 $n$。
- 第二行：$n$ 个点权 $v_i$。
- 接下来 $n$ 行：节点 $i$ 的左右孩子编号 $l_i, r_i$；不存在则为 `-1`。
- 节点 `1` 是整棵树的根。

输出：最大对称二叉子树的节点数。

## 4.2 思路分析

直接枚举每个节点，再递归比较其左右子树，最坏可能反复访问同一批节点。更稳妥的方式是给每棵子树做“结构指纹”。

定义两个编号：

- `normal_id[u]`：以 `u` 为根的原方向子树编号；
- `mirror_id[u]`：以 `u` 为根的镜像方向子树编号。

若一棵子树对称，则它的原方向结构与镜像方向结构完全相同：

```text
normal_id[u] == mirror_id[u]
```

为了避免字符串巨大，我们用字典给三元组编号：

```text
(value[u], left_id, right_id) -> unique_id
```

后序处理每个节点，先知道孩子编号，再计算当前节点编号。

## 4.3 参考代码

```python
import sys

input = sys.stdin.readline

n = int(input())
value = [0] + list(map(int, input().split()))
left = [0] * (n + 1)
right = [0] * (n + 1)

for i in range(1, n + 1):
    l, r = map(int, input().split())
    left[i] = 0 if l == -1 else l
    right[i] = 0 if r == -1 else r

# 迭代后序，避免深树递归爆栈
order = []
stack = [1]
while stack:
    u = stack.pop()
    if u == 0:
        continue
    order.append(u)
    stack.append(left[u])
    stack.append(right[u])

normal_id = [0] * (n + 1)
mirror_id = [0] * (n + 1)
size = [0] * (n + 1)

ids = {}
next_id = 1
best = 1


def get_id(key):
    global next_id
    if key not in ids:
        ids[key] = next_id
        next_id += 1
    return ids[key]

for u in reversed(order):
    l, r = left[u], right[u]
    size[u] = size[l] + size[r] + 1

    normal_id[u] = get_id((value[u], normal_id[l], normal_id[r]))
    mirror_id[u] = get_id((value[u], mirror_id[r], mirror_id[l]))

    if normal_id[u] == mirror_id[u]:
        best = max(best, size[u])

print(best)
```

复杂度：每个节点处理一次，时间 $O(n)$，空间 $O(n)$。这种写法的核心优点是把“递归比较整棵子树”压缩成了“比较两个编号”。

## 4.4 变体讨论

### 变体 1：只判断整棵树是否对称

如果只问整棵树是否对称，不需要子树哈希，直接用栈成对比较即可。

```python
def is_whole_tree_symmetric(value, left, right) -> bool:
    stack = [(left[1], right[1])]

    while stack:
        a, b = stack.pop()
        if a == 0 and b == 0:
            continue
        if a == 0 or b == 0:
            return False
        if value[a] != value[b]:
            return False

        stack.append((left[a], right[b]))
        stack.append((right[a], left[b]))

    return True
```

### 变体 2：小数据下的记忆化镜像比较

当 $n$ 很小，直接写镜像判断更容易理解。用 `lru_cache` 可以避免重复比较同一对节点。

```python
from functools import lru_cache


def largest_symmetric_small(value, left, right) -> int:
    n = len(value) - 1

    @lru_cache(None)
    def same(a: int, b: int) -> bool:
        if a == 0 and b == 0:
            return True
        if a == 0 or b == 0:
            return False
        return (
            value[a] == value[b]
            and same(left[a], right[b])
            and same(right[a], left[b])
        )

    def subtree_size(u: int) -> int:
        if u == 0:
            return 0
        return 1 + subtree_size(left[u]) + subtree_size(right[u])

    best = 1
    for u in range(1, n + 1):
        if same(left[u], right[u]):
            best = max(best, subtree_size(u))
    return best
```

## 4.5 小结

对称二叉树的本质是“左子树与右子树互为镜像”。小数据可以直接递归比较；大数据更适合用后序 DP 或子树哈希，把结构相等转化为编号相等。

---

# 题目五：P1364 医院设置

[Luogu P1364](https://www.luogu.com.cn/problem/P1364)

## 5.1 题目描述

给定一棵二叉树，每个节点有居民人口。现在选择一个节点建立医院，使所有居民到医院的距离总和最小。

- 相邻节点距离为 $1$。
- 输入第一行是节点数 $n$。
- 接下来 $n$ 行：`w u v`，表示当前节点人口为 `w`，左孩子为 `u`，右孩子为 `v`；`0` 表示没有孩子。
- 输出最小距离和。
- $1 \le n \le 100$。

## 5.2 思路分析

虽然输入是二叉树，但“居民去医院”可以沿父子边双向移动，所以计算距离时要把它看成一棵无向树。

朴素做法：枚举每个医院位置，做一次 DFS/BFS 求距离和。因为 $n \le 100$，完全可行。

更有教学价值的做法是换根 DP。

设：

- `pop[u]`：节点 `u` 的人口；
- `sub[u]`：以 `u` 为根的子树人口总和；
- `dp[u]`：医院设在 `u` 时，全树距离加权和。

先以 `1` 为根做 DFS，得到：

```text
dp[1] = sum(pop[x] * depth[x])
```

再考虑从 `u` 换根到它的孩子 `v`：

- `v` 子树内所有居民离医院近了 1，总贡献减少 `sub[v]`；
- 其余居民离医院远了 1，总贡献增加 `total_pop - sub[v]`。

所以：

$$
dp[v] = dp[u] - sub[v] + (total - sub[v]) = dp[u] + total - 2 \cdot sub[v]
$$

## 5.3 参考代码

```python
import sys

input = sys.stdin.readline

n = int(input())
pop = [0] * (n + 1)
g = [[] for _ in range(n + 1)]

for i in range(1, n + 1):
    w, l, r = map(int, input().split())
    pop[i] = w
    if l:
        g[i].append(l)
        g[l].append(i)
    if r:
        g[i].append(r)
        g[r].append(i)

total = sum(pop)
sub = [0] * (n + 1)
dp = [0] * (n + 1)


def dfs1(u: int, parent: int, depth: int) -> None:
    sub[u] = pop[u]
    dp[1] += pop[u] * depth
    for v in g[u]:
        if v == parent:
            continue
        dfs1(v, u, depth + 1)
        sub[u] += sub[v]


def dfs2(u: int, parent: int) -> None:
    for v in g[u]:
        if v == parent:
            continue
        dp[v] = dp[u] + total - 2 * sub[v]
        dfs2(v, u)


dfs1(1, 0, 0)
dfs2(1, 0)

print(min(dp[1:]))
```

复杂度：时间 $O(n)$，空间 $O(n)$。

## 5.4 变体讨论

### 变体 1：枚举医院位置 + BFS

对小数据，这是最直观、最不容易写错的方法。

```python
from collections import deque


def solve_by_bfs(pop: list[int], g: list[list[int]]) -> int:
    n = len(pop) - 1
    best = 10**30

    for start in range(1, n + 1):
        dist = [-1] * (n + 1)
        dist[start] = 0
        q = deque([start])

        while q:
            u = q.popleft()
            for v in g[u]:
                if dist[v] == -1:
                    dist[v] = dist[u] + 1
                    q.append(v)

        cost = sum(pop[i] * dist[i] for i in range(1, n + 1))
        best = min(best, cost)

    return best
```

### 变体 2：边权不全为 1 的医院设置

若每条边有长度，换根公式改成：从 `u` 移到孩子 `v`，距离变化量是边权 `w`。

```python
def reroot_weighted(pop: list[int], g: list[list[tuple[int, int]]]) -> int:
    n = len(pop) - 1
    total = sum(pop)
    sub = [0] * (n + 1)
    dp = [0] * (n + 1)

    def dfs1(u: int, parent: int, dist: int) -> None:
        sub[u] = pop[u]
        dp[1] += pop[u] * dist
        for v, w in g[u]:
            if v == parent:
                continue
            dfs1(v, u, dist + w)
            sub[u] += sub[v]

    def dfs2(u: int, parent: int) -> None:
        for v, w in g[u]:
            if v == parent:
                continue
            dp[v] = dp[u] + (total - 2 * sub[v]) * w
            dfs2(v, u)

    dfs1(1, 0, 0)
    dfs2(1, 0)
    return min(dp[1:])
```

## 5.5 小结

P1364 的核心不是“二叉树遍历”，而是“树上距离和”。枚举 BFS 适合小数据；换根 DP 适合把复杂度从 $O(n^2)$ 降到 $O(n)$，也是树形 DP 中非常经典的思想。

---

# 本节总小结

| 题目 | 关键句 |
|---|---|
| P1030 | 中序切左右，后序找根，递归得到先序。 |
| P1229 | 前序 + 后序缺少左右信息，单孩子节点带来 $\times 2$。 |
| P1305 | 先把输入转成左右孩子表，再套遍历模板。 |
| P5018 | 对称等价于原树编号与镜像编号相同。 |
| P1364 | 树上距离和可用换根公式 `dp[v] = dp[u] + total - 2 * sub[v]`。 |

课堂上建议把这 5 题串成一条线：

1. 从遍历恢复树；
2. 理解遍历信息何时不充分；
3. 熟悉树的输入表示；
4. 用递归/哈希判断结构性质；
5. 在树上维护子树信息并换根。
